In [1]:
import os
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms

try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline

### Зафиксируем seed для воспроизводимости

In [2]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

### Задаем параметры (конфиг)

In [3]:
all_brands = ('Audi', 'BMW', 'Chevrolet', 'Hyundai', 'Kia', 'Lexus', 'Mercedes-Benz', 'Toyota')
all_body_types = ('sedan', 'SUV')

class CFG:
  api = ""
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 10
  train_batch_size = 32
  test_batch_size = 512
  num_workers = 0
  lr = 3e-4
  seed = 42
  classes = all_body_types
  wandb = True

In [ ]:
# вход в W&B (ключ спросит один раз, берём с https://wandb.ai/authorize)
wandb.login()


In [4]:
seed_everything(CFG.seed)

In [5]:
# Переведем наш класс с параметрами в словарь

def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

### Импортируем наш датасет с картинками

In [6]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  !cp -r /content/drive/MyDrive/gp5_images/data /content/data

  base_dir = '/content/data'
except:
  base_dir = '../data'

from pathlib import Path

base_dir = Path(base_dir)

In [7]:
df_bt = pd.read_csv(base_dir / 'prepared_df_images.csv')
df_bt

,kolesa_id,body_type,img_filename,target
0,220599328,Седан,220599328.webp,0
1,222486528,Седан,222486528.webp,0
2,218695550,Внедорожник,218695550.webp,1
3,214781208,Седан,214781208.webp,0
4,222346737,Седан,222346737.webp,0
...,...,...,...,...
4428,222478046,Седан,222478046.webp,0
4429,222704181,Седан,222704181.webp,0
4430,220000872,Седан,220000872.webp,0
4431,222542135,Седан,222542135.webp,0


In [8]:
df_brand = pd.read_csv(base_dir / 'brand_df_images.csv')
df_brand

,kolesa_id,brand,body_type,img_filename,target
0,220599328,Chevrolet,Седан,220599328.webp,2
1,222486528,Chevrolet,Седан,222486528.webp,2
2,218695550,Chevrolet,Внедорожник,218695550.webp,2
3,214781208,Chevrolet,Седан,214781208.webp,2
4,222346737,Chevrolet,Седан,222346737.webp,2
...,...,...,...,...,...
4428,222478046,Hyundai,Седан,222478046.webp,3
4429,222704181,Hyundai,Седан,222704181.webp,3
4430,220000872,Hyundai,Седан,220000872.webp,3
4431,222542135,Hyundai,Седан,222542135.webp,3


In [9]:
train_df_bt, test_all_df_bt = train_test_split(df_bt, test_size=0.3, stratify=df_bt['target'], random_state=CFG.seed)
val_df_bt, test_df_bt = train_test_split(test_all_df_bt, test_size=0.5, stratify=test_all_df_bt['target'], random_state=CFG.seed)

train_df_brand, test_all_df_brand = train_test_split(df_brand, test_size=0.3, stratify=df_brand['target'], random_state=CFG.seed)
val_df_brand, test_df_brand = train_test_split(test_all_df_brand, test_size=0.5, stratify=test_all_df_brand['target'], random_state=CFG.seed)

In [10]:
# https://stackoverflow.com/questions/65231299/load-csv-and-image-dataset-in-pytorch

class KolesaDataset(torch.utils.data.Dataset):
    def __init__(self, df, images_folder = base_dir / 'images', transform = None):
        # т.к. после train_test_split индексы сохряняются из исходного датасета, то мы их сбрасываем и удаляем колонку со старыми индексами для корректной работы __getitem___
        self.df = df.reset_index().drop(columns=['index'])
        self.images_folder = images_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        filename = self.df.loc[index, 'img_filename']
        label = self.df.loc[index, 'target']
        image = Image.open(os.path.join(self.images_folder, filename)).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, label



### Функции обучения и инференса

In [11]:
# функция обучения
def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()
    train_loss = 0
    correct = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # обнуляем градиенты
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target) # считаем лосс
        train_loss.backward() # обратный проход
        optimizer.step() # делаем шаг оптимизатором

    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))

    # логируем функцию потерь и точность
    if WANDB:
        wandb.log({'train_loss': train_loss,
                   'train_accuracy': correct / len(train_loader.dataset)})

In [12]:
# функция инференса
def test(model, device, test_loader, criterion, WANDB):
    model.eval()
    test_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            # https://stackoverflow.com/questions/53467215/convert-pytorch-cuda-tensor-to-numpy-array
            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Test set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        test_loss, 100. * correct / len(test_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'test_loss': test_loss,
                   'test_accuracy': correct / len(test_loader.dataset)})

# функция инференса
def val(model, device, val_loader, criterion, WANDB):
    model.eval()
    val_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Val set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        val_loss, 100. * correct / len(val_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'val_loss': val_loss,
                   'val_accuracy': correct / len(val_loader.dataset)})

### Основная функция для прогона моделей

In [13]:
def main_kolesa(model, train_df, val_df):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))
        # логируем архитектуру модели
        # https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})

    seed_everything(CFG.seed)
    # https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
    # т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # наши картинки в исходном размере 750*470 + если домножить на число каналов (3), то получится около миллиарда параметра на вход.
    # Это довольно много так что пропорционально уменьшим размерность. У нас 750 к 470 это, можно сказать, 1.6 к 1. Уменьшим 750 до 160 => 470 -> 96.
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    # test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    # test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)

    print('Training is ended!')

### Сохранение моделей

Чтобы у каждой модели сохранялся артефакт, добавим маленькую вспомогательную функцию. `main_kolesa` логирует loss/accuracy по эпохам в W&B сам, а эта функция дописывает в тот же запуск файл с весами.

In [14]:
def save_model_artifact(model, name):
    # сохранение модели как артефакт
    # https://docs.wandb.ai/guides/artifacts
    path = f'../data/models/{name}.pt' #путь к файлу в зависимости от имени модели (передаем в начале), .pt -расширение для объектов pytorch
    torch.save(model.state_dict(), path) #сохраняем параметры модели
    if CFG.wandb:
        wandb.run.name = name   # называем запуск как модель
        artifact = wandb.Artifact(name, type='model') #создаем артефакт-контейннер
        artifact.add_file(path) #добавляем в него файл с параметрами модели (брали выше)
        wandb.log_artifact(artifact) #загружаем артефакт и привящываем к текущему щапуску
        wandb.finish()  # закрываем текущий запуск
    print(f'[OK] модель сохранена: {path}')

In [ ]:
# логируем данные в W&B, чтобы запуск воспроизводился из неё
# https://docs.wandb.ai/guides/artifacts
# выполнить ОДИН раз
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, name='dataset')
    art = wandb.Artifact('kolesa-data', type='dataset')
    art.add_file('../data/prepared_df_images.csv')
    art.add_file('../data/brand_df_images.csv')
    art.add_dir('../data/images', name='images')   # ~475 МБ, грузится долго
    wandb.log_artifact(art)
    wandb.finish()


## Fully Connected в качестве baseline

Возьмем для примера определение весов нормальным распределением

In [15]:
# определяем полносвязанную сеть
class FC_Net(nn.Module):
    def __init__(self, task='body_type'):
        super(FC_Net, self).__init__()

        self.task = task

        self.fc1 = nn.Linear(160*96*3, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 128)
        if self.task == 'body_type':
            self.fc4 = nn.Linear(128, 2)
        elif self.task == 'brand':
            self.fc4 = nn.Linear(128, 8)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.view(-1, 160*96*3)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

#### Задача на тип кузова

In [16]:
model_baseline_bt = FC_Net()

In [17]:
summary(model=model_baseline_bt,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
FC_Net                                   [32, 3, 96, 160]     [32, 2]              --                   True
├─Linear: 1-1                            [32, 46080]          [32, 1024]           47,186,944           True
├─Linear: 1-2                            [32, 1024]           [32, 512]            524,800              True
├─Linear: 1-3                            [32, 512]            [32, 128]            65,664               True
├─Linear: 1-4                            [32, 128]            [32, 2]              258                  True
Total params: 47,777,666
Trainable params: 47,777,666
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.53
Input size (MB): 5.90
Forward/backward pass size (MB): 0.43
Params size (MB): 191.11
Estimated Total Size (MB): 197.44

In [18]:
main_kolesa(model_baseline_bt, train_df = train_df_bt, val_df = val_df_bt)
save_model_artifact(model_baseline_bt, 'BaseLine_FC_body_type')


Epoch: 1


100%|██████████| 97/97 [00:13<00:00,  7.09it/s]



Train set: Average loss: 200857.7031, Accuracy: 54.04%
Val set: Average loss: 133297.0781, Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.58      0.62       418
         SUV       0.41      0.50      0.45       247

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.53       665
weighted avg       0.57      0.55      0.55       665


Epoch: 2


100%|██████████| 97/97 [00:12<00:00,  7.49it/s]



Train set: Average loss: 100811.1328, Accuracy: 56.91%
Val set: Average loss: 111838.9688, Accuracy: 56.24%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.63      0.64       418
         SUV       0.42      0.45      0.43       247

    accuracy                           0.56       665
   macro avg       0.54      0.54      0.54       665
weighted avg       0.57      0.56      0.57       665


Epoch: 3


...


Train set: Average loss: 67743.3906, Accuracy: 62.84%
Val set: Average loss: 82431.4219, Accuracy: 57.29%

Classification report:
              precision    recall  f1-score   support

       sedan       0.67      0.63      0.65       418
         SUV       0.43      0.48      0.46       247

    accuracy                           0.57       665
   macro avg       0.55      0.55      0.55       665
weighted avg       0.58      0.57      0.58       665


Epoch: 9


100%|██████████| 97/97 [00:12<00:00,  7.58it/s]



Train set: Average loss: 51832.0078, Accuracy: 64.90%
Val set: Average loss: 74938.6016, Accuracy: 56.69%

Classification report:
              precision    recall  f1-score   support

       sedan       0.68      0.59      0.63       418
         SUV       0.43      0.52      0.47       247

    accuracy                           0.57       665
   macro avg       0.55      0.56      0.55       665
weighted avg       0.59      0.57      0.57       665


Epoch: 10


100%|██████████| 97/97 [00:12<00:00,  7.72it/s]



Train set: Average loss: 30968.8867, Accuracy: 63.68%
Val set: Average loss: 71612.6641, Accuracy: 55.49%

Classification report:
              precision    recall  f1-score   support

       sedan       0.67      0.57      0.62       418
         SUV       0.42      0.53      0.47       247

    accuracy                           0.55       665
   macro avg       0.55      0.55      0.54       665
weighted avg       0.58      0.55      0.56       665

Training is ended!
[OK] модель сохранена: ../data/models/BaseLine_FC_body_type.pt


#### Задача на определение марки

In [19]:
CFG.classes = all_brands

model_baseline_brand = FC_Net(task='brand')

In [20]:
summary(model=model_baseline_brand,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
FC_Net                                   [32, 3, 96, 160]     [32, 8]              --                   True
├─Linear: 1-1                            [32, 46080]          [32, 1024]           47,186,944           True
├─Linear: 1-2                            [32, 1024]           [32, 512]            524,800              True
├─Linear: 1-3                            [32, 512]            [32, 128]            65,664               True
├─Linear: 1-4                            [32, 128]            [32, 8]              1,032                True
Total params: 47,778,440
Trainable params: 47,778,440
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.53
Input size (MB): 5.90
Forward/backward pass size (MB): 0.43
Params size (MB): 191.11
Estimated Total Size (MB): 197.44

In [21]:
main_kolesa(model_baseline_brand, train_df = train_df_brand, val_df = val_df_brand)
save_model_artifact(model_baseline_brand, 'BaseLine_FC_brand')


Epoch: 1


100%|██████████| 97/97 [00:13<00:00,  7.38it/s]



Train set: Average loss: 285470.6250, Accuracy: 13.89%
Val set: Average loss: 388377.4688, Accuracy: 12.18%

Classification report:
               precision    recall  f1-score   support

         Audi       0.08      0.10      0.09        73
          BMW       0.17      0.09      0.12        90
    Chevrolet       0.15      0.19      0.17        79
      Hyundai       0.11      0.13      0.12        85
          Kia       0.16      0.16      0.16        77
        Lexus       0.13      0.13      0.13        95
Mercedes-Benz       0.10      0.11      0.11        81
       Toyota       0.08      0.08      0.08        85

     accuracy                           0.12       665
    macro avg       0.12      0.12      0.12       665
 weighted avg       0.13      0.12      0.12       665


Epoch: 2


100%|██████████| 97/97 [00:12<00:00,  7.89it/s]



Train set: Average loss: 319479.7188, Accuracy: 13.63%
Val set: Average loss: 333627.9688, Accuracy: 11.28%

Classification report:
               precision    recall  f1-score   support

         Audi       0.07      0.07      0.07        73
          BMW       0.14      0.12      0.13        90
    Chevrolet       0.13      0.15      0.14        79
      Hyundai       0.11      0.12      0.12        85
          Kia       0.13      0.10      0.12        77
        Lexus       0.09      0.12      0.10        95
Mercedes-Benz       0.14      0.11      0.12        81
       Toyota       0.10      0.11      0.11        85

     accuracy                           0.11       665
    macro avg       0.11      0.11      0.11       665
 weighted avg       0.11      0.11      0.11       665


Epoch: 3


...


Train set: Average loss: 75338.2891, Accuracy: 17.82%
Val set: Average loss: 143131.1406, Accuracy: 12.93%

Classification report:
               precision    recall  f1-score   support

         Audi       0.09      0.08      0.08        73
          BMW       0.15      0.12      0.13        90
    Chevrolet       0.15      0.18      0.16        79
      Hyundai       0.06      0.05      0.05        85
          Kia       0.12      0.12      0.12        77
        Lexus       0.14      0.17      0.15        95
Mercedes-Benz       0.14      0.14      0.14        81
       Toyota       0.16      0.18      0.17        85

     accuracy                           0.13       665
    macro avg       0.13      0.13      0.13       665
 weighted avg       0.13      0.13      0.13       665


Epoch: 9


100%|██████████| 97/97 [00:12<00:00,  7.97it/s]



Train set: Average loss: 46430.6680, Accuracy: 16.89%
Val set: Average loss: 120568.9688, Accuracy: 14.14%

Classification report:
               precision    recall  f1-score   support

         Audi       0.09      0.08      0.09        73
          BMW       0.15      0.16      0.15        90
    Chevrolet       0.17      0.22      0.19        79
      Hyundai       0.07      0.06      0.06        85
          Kia       0.16      0.16      0.16        77
        Lexus       0.17      0.19      0.18        95
Mercedes-Benz       0.18      0.15      0.16        81
       Toyota       0.11      0.12      0.11        85

     accuracy                           0.14       665
    macro avg       0.14      0.14      0.14       665
 weighted avg       0.14      0.14      0.14       665


Epoch: 10


100%|██████████| 97/97 [00:12<00:00,  7.98it/s]



Train set: Average loss: 16634.3750, Accuracy: 17.24%
Val set: Average loss: 97859.7031, Accuracy: 14.14%

Classification report:
               precision    recall  f1-score   support

         Audi       0.09      0.10      0.09        73
          BMW       0.14      0.11      0.13        90
    Chevrolet       0.17      0.16      0.17        79
      Hyundai       0.10      0.09      0.10        85
          Kia       0.18      0.14      0.16        77
        Lexus       0.18      0.20      0.19        95
Mercedes-Benz       0.16      0.12      0.14        81
       Toyota       0.12      0.19      0.14        85

     accuracy                           0.14       665
    macro avg       0.14      0.14      0.14       665
 weighted avg       0.14      0.14      0.14       665

Training is ended!
[OK] модель сохранена: ../data/models/BaseLine_FC_brand.pt


Модель сильно переобучилась. Посмотрим далее сверточную сеть

## Простой CNN

In [22]:
class CNN_Net(nn.Module):
    def __init__(self, task='body_type'):
        super(CNN_Net, self).__init__()

        self.task = task

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2) # 48 80

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2) # 24 40

        self.conv3 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2) # 12 20

        self.fc1   = torch.nn.Linear(12 * 20 * 64, 256)
        self.act4  = torch.nn.Tanh()

        self.fc2   = torch.nn.Linear(256, 64)
        self.act5  = torch.nn.Tanh()
        
        if task == 'body_type':
            self.fc3   = torch.nn.Linear(64, 2)
        elif task == 'brand':
            self.fc3   = torch.nn.Linear(64, 8)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.act4(x)
        x = self.fc2(x)
        x = self.act5(x)
        x = self.fc3(x)

        return x

#### Задача на тип кузова

In [23]:
CFG.classes = all_body_types

model_cnn_1_bt = CNN_Net()

In [24]:
summary(model=model_cnn_1_bt,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNN_Net                                  [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 16, 96, 160]    448                  True
├─ReLU: 1-2                              [32, 16, 96, 160]    [32, 16, 96, 160]    --                   --
├─MaxPool2d: 1-3                         [32, 16, 96, 160]    [32, 16, 48, 80]     --                   --
├─Conv2d: 1-4                            [32, 16, 48, 80]     [32, 32, 48, 80]     4,640                True
├─ReLU: 1-5                              [32, 32, 48, 80]     [32, 32, 48, 80]     --                   --
├─MaxPool2d: 1-6                         [32, 32, 48, 80]     [32, 32, 24, 40]     --                   --
├─Conv2d: 1-7                            [32, 32, 24, 40]     [32, 64, 24, 40]     18,496               True
├─ReLU: 1-8           

In [25]:
main_kolesa(model_cnn_1_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_cnn_1_bt, 'Base_CNN_body_type')


Epoch: 1


100%|██████████| 97/97 [00:12<00:00,  7.56it/s]



Train set: Average loss: 0.7569, Accuracy: 62.20%
Val set: Average loss: 0.6640, Accuracy: 63.61%

Classification report:
              precision    recall  f1-score   support

       sedan       0.63      1.00      0.78       418
         SUV       1.00      0.02      0.04       247

    accuracy                           0.64       665
   macro avg       0.82      0.51      0.41       665
weighted avg       0.77      0.64      0.50       665


Epoch: 2


100%|██████████| 97/97 [00:13<00:00,  7.45it/s]



Train set: Average loss: 0.6975, Accuracy: 63.26%
Val set: Average loss: 0.6970, Accuracy: 56.69%

Classification report:
              precision    recall  f1-score   support

       sedan       0.74      0.48      0.58       418
         SUV       0.45      0.71      0.55       247

    accuracy                           0.57       665
   macro avg       0.59      0.60      0.57       665
weighted avg       0.63      0.57      0.57       665


Epoch: 3


...


Train set: Average loss: 0.6542, Accuracy: 70.51%
Val set: Average loss: 0.6426, Accuracy: 65.56%

Classification report:
              precision    recall  f1-score   support

       sedan       0.74      0.70      0.72       418
         SUV       0.53      0.58      0.56       247

    accuracy                           0.66       665
   macro avg       0.64      0.64      0.64       665
weighted avg       0.66      0.66      0.66       665


Epoch: 9


100%|██████████| 97/97 [00:13<00:00,  7.25it/s]



Train set: Average loss: 0.5823, Accuracy: 71.87%
Val set: Average loss: 0.6666, Accuracy: 64.06%

Classification report:
              precision    recall  f1-score   support

       sedan       0.75      0.65      0.69       418
         SUV       0.51      0.63      0.56       247

    accuracy                           0.64       665
   macro avg       0.63      0.64      0.63       665
weighted avg       0.66      0.64      0.65       665


Epoch: 10


100%|██████████| 97/97 [00:13<00:00,  7.39it/s]



Train set: Average loss: 0.6147, Accuracy: 72.09%
Val set: Average loss: 0.6598, Accuracy: 66.02%

Classification report:
              precision    recall  f1-score   support

       sedan       0.77      0.66      0.71       418
         SUV       0.53      0.66      0.59       247

    accuracy                           0.66       665
   macro avg       0.65      0.66      0.65       665
weighted avg       0.68      0.66      0.67       665

Training is ended!
[OK] модель сохранена: ../data/models/Base_CNN_body_type.pt


#### Задача на определение марки

In [26]:
CFG.classes = all_brands

model_cnn_1_brand = CNN_Net(task='brand')

In [27]:
summary(model=model_cnn_1_brand,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNN_Net                                  [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 16, 96, 160]    448                  True
├─ReLU: 1-2                              [32, 16, 96, 160]    [32, 16, 96, 160]    --                   --
├─MaxPool2d: 1-3                         [32, 16, 96, 160]    [32, 16, 48, 80]     --                   --
├─Conv2d: 1-4                            [32, 16, 48, 80]     [32, 32, 48, 80]     4,640                True
├─ReLU: 1-5                              [32, 32, 48, 80]     [32, 32, 48, 80]     --                   --
├─MaxPool2d: 1-6                         [32, 32, 48, 80]     [32, 32, 24, 40]     --                   --
├─Conv2d: 1-7                            [32, 32, 24, 40]     [32, 64, 24, 40]     18,496               True
├─ReLU: 1-8           

In [28]:
main_kolesa(model_cnn_1_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_cnn_1_brand, 'Base_CNN_brand')


Epoch: 1


100%|██████████| 97/97 [00:13<00:00,  7.23it/s]



Train set: Average loss: 2.0729, Accuracy: 14.24%


/Users/alexander/Documents/Study/СМАДИМО/gp5/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/alexander/Documents/Study/СМАДИМО/gp5/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/alexander/Documents/Study/СМАДИМО/gp5/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

Val set: Average loss: 2.0768, Accuracy: 14.14%

Classification report:
               precision    recall  f1-score   support

         Audi       0.00      0.00      0.00        73
          BMW       0.11      0.33      0.17        90
    Chevrolet       0.21      0.23      0.22        79
      Hyundai       0.00      0.00      0.00        85
          Kia       0.23      0.17      0.20        77
        Lexus       0.17      0.24      0.20        95
Mercedes-Benz       0.04      0.01      0.02        81
       Toyota       0.09      0.11      0.10        85

     accuracy                           0.14       665
    macro avg       0.11      0.14      0.11       665
 weighted avg       0.11      0.14      0.11       665


Epoch: 2


...


Train set: Average loss: 1.8840, Accuracy: 28.07%
Val set: Average loss: 2.1148, Accuracy: 19.40%

Classification report:
               precision    recall  f1-score   support

         Audi       0.15      0.14      0.14        73
          BMW       0.21      0.39      0.27        90
    Chevrolet       0.35      0.18      0.24        79
      Hyundai       0.12      0.01      0.02        85
          Kia       0.25      0.13      0.17        77
        Lexus       0.23      0.08      0.12        95
Mercedes-Benz       0.18      0.44      0.26        81
       Toyota       0.14      0.18      0.16        85

     accuracy                           0.19       665
    macro avg       0.20      0.19      0.17       665
 weighted avg       0.20      0.19      0.17       665


Epoch: 9


100%|██████████| 97/97 [00:12<00:00,  7.52it/s]



Train set: Average loss: 1.7233, Accuracy: 31.20%
Val set: Average loss: 2.1486, Accuracy: 18.20%

Classification report:
               precision    recall  f1-score   support

         Audi       0.12      0.12      0.12        73
          BMW       0.22      0.39      0.28        90
    Chevrolet       0.36      0.19      0.25        79
      Hyundai       0.00      0.00      0.00        85
          Kia       0.23      0.17      0.20        77
        Lexus       0.17      0.11      0.13        95
Mercedes-Benz       0.15      0.33      0.20        81
       Toyota       0.14      0.14      0.14        85

     accuracy                           0.18       665
    macro avg       0.17      0.18      0.16       665
 weighted avg       0.17      0.18      0.16       665


Epoch: 10


100%|██████████| 97/97 [00:12<00:00,  7.70it/s]



Train set: Average loss: 1.5459, Accuracy: 33.23%
Val set: Average loss: 2.1410, Accuracy: 19.10%

Classification report:
               precision    recall  f1-score   support

         Audi       0.17      0.22      0.19        73
          BMW       0.22      0.48      0.30        90
    Chevrolet       0.50      0.16      0.25        79
      Hyundai       0.10      0.01      0.02        85
          Kia       0.22      0.18      0.20        77
        Lexus       0.19      0.08      0.12        95
Mercedes-Benz       0.15      0.26      0.19        81
       Toyota       0.12      0.13      0.12        85

     accuracy                           0.19       665
    macro avg       0.21      0.19      0.17       665
 weighted avg       0.21      0.19      0.17       665

Training is ended!
[OK] модель сохранена: ../data/models/Base_CNN_brand.pt


Не сильно далеко ушли от FC, по крайней мере, пока качество не самое приятное

## CNN с регуляризацией

Теперь попробуем добавить регуляризацию, а именно известные нам BatchNorm и Dropout.

Также поменяем функции активации https://deepmachinelearning.ru/docs/Neural-networks/MLP/Activation-functions. В частности попробуем LeakyReLU.

И наконец добавим побольше сверточных слоев.

In [29]:
class CNNReg_Net(nn.Module):
    def __init__(self, task='body_type'):
        super(CNNReg_Net, self).__init__()

        self.task = task

        self.conv3 = torch.nn.Conv2d(3, 32, 3, padding=1)
        self.bn3   = torch.nn.BatchNorm2d(32)
        self.act3  = torch.nn.LeakyReLU()

        self.conv4 = torch.nn.Conv2d(32, 32, 3, padding=1)
        self.bn4   = torch.nn.BatchNorm2d(32)
        self.act4  = torch.nn.LeakyReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2) # 24 40

        self.conv5 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.bn5   = torch.nn.BatchNorm2d(64)
        self.act5  = torch.nn.LeakyReLU()

        self.conv6 = torch.nn.Conv2d(64, 64, 3, padding=1)
        self.bn6   = torch.nn.BatchNorm2d(64)
        self.act6  = torch.nn.LeakyReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2) # 12 20

        self.drop1 = torch.nn.Dropout2d(0.10)

        self.conv7 = torch.nn.Conv2d(64, 128, 3, padding=1)
        self.bn7   = torch.nn.BatchNorm2d(128)
        self.act7  = torch.nn.LeakyReLU()
        self.drop2 = torch.nn.Dropout2d(0.15)

        self.conv8 = torch.nn.Conv2d(128, 128, 3, padding=1)
        self.bn8   = torch.nn.BatchNorm2d(128)
        self.act8  = torch.nn.LeakyReLU()
        self.pool4 = torch.nn.MaxPool2d(2, 2)
        self.drop3 = torch.nn.Dropout2d(0.2)

        self.conv9 = torch.nn.Conv2d(128, 256, 3, padding=1)
        self.bn9   = torch.nn.BatchNorm2d(256)
        self.act9  = torch.nn.LeakyReLU()
        self.pool5 = torch.nn.MaxPool2d(2, 2)
        self.drop4 = torch.nn.Dropout2d(0.25)

        self.fc1   = torch.nn.Linear(6 * 10 * 256, 256)
        self.bn10  = torch.nn.BatchNorm1d(256)
        self.act10 = torch.nn.LeakyReLU()
        self.drop5 = torch.nn.Dropout(0.3)

        self.fc2   = torch.nn.Linear(256, 64)
        self.act11 = torch.nn.LeakyReLU()
        self.drop6 = torch.nn.Dropout(0.2)

        if task == 'body_type':
            self.fc3 = torch.nn.Linear(64, 2)
        elif task == 'brand':
            self.fc3 = torch.nn.Linear(64, 8)
            

    def forward(self, x):
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)
        x = self.pool2(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.conv6(x)
        x = self.bn6(x)
        x = self.act6(x)
        x = self.pool3(x)
        x = self.drop1(x)

        x = self.conv7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop2(x)

        x = self.conv8(x)
        x = self.bn8(x)
        x = self.act8(x)
        x = self.pool4(x)
        x = self.drop3(x)

        x = self.conv9(x)
        x = self.bn9(x)
        x = self.act9(x)
        x = self.pool5(x)
        x = self.drop4(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.bn10(x)
        x = self.act10(x)
        x = self.drop5(x)

        x = self.fc2(x)
        x = self.act11(x)
        x = self.drop6(x)

        x = self.fc3(x)

        return x

На этой модели попробуем побольше эпох

In [30]:
CFG.num_epochs = 30

#### Задача на тип кузова

In [31]:
CFG.classes = all_body_types

model_cnn_2_bt = CNNReg_Net()

In [32]:
summary(model=model_cnn_2_bt,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNNReg_Net                               [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 32, 96, 160]    896                  True
├─BatchNorm2d: 1-2                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-3                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─Conv2d: 1-4                            [32, 32, 96, 160]    [32, 32, 96, 160]    9,248                True
├─BatchNorm2d: 1-5                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-6                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─MaxPool2d: 1-7                         [32, 32, 96, 160]    [32, 32, 48, 80]     --                   --
├─Conv2d: 1-8       

In [33]:
main_kolesa(model_cnn_2_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_cnn_2_bt, 'Reg_CNN_body_type')


Epoch: 1


100%|██████████| 97/97 [00:15<00:00,  6.35it/s]



Train set: Average loss: 0.8110, Accuracy: 61.26%
Val set: Average loss: 0.6159, Accuracy: 65.41%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.92      0.77       418
         SUV       0.60      0.21      0.31       247

    accuracy                           0.65       665
   macro avg       0.63      0.56      0.54       665
weighted avg       0.64      0.65      0.60       665


Epoch: 2


100%|██████████| 97/97 [00:15<00:00,  6.26it/s]



Train set: Average loss: 0.7646, Accuracy: 64.00%
Val set: Average loss: 0.6254, Accuracy: 66.02%

Classification report:
              precision    recall  f1-score   support

       sedan       0.67      0.92      0.77       418
         SUV       0.62      0.21      0.32       247

    accuracy                           0.66       665
   macro avg       0.64      0.57      0.55       665
weighted avg       0.65      0.66      0.60       665


Epoch: 3


...


Train set: Average loss: 0.4263, Accuracy: 84.43%
Val set: Average loss: 0.7434, Accuracy: 68.27%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.60      0.70       418
         SUV       0.55      0.82      0.66       247

    accuracy                           0.68       665
   macro avg       0.70      0.71      0.68       665
weighted avg       0.74      0.68      0.69       665


Epoch: 29


100%|██████████| 97/97 [00:15<00:00,  6.28it/s]



Train set: Average loss: 0.5795, Accuracy: 84.53%
Val set: Average loss: 0.6479, Accuracy: 70.68%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.65      0.74       418
         SUV       0.58      0.80      0.67       247

    accuracy                           0.71       665
   macro avg       0.71      0.73      0.70       665
weighted avg       0.75      0.71      0.71       665


Epoch: 30


  9%|▉         | 9/97 [00:01<00:15,  5.74it/s]


KeyboardInterrupt: 

#### Задача на определение марки

In [ ]:
CFG.classes = all_brands

model_cnn_2_brand = CNNReg_Net(task='brand')

In [ ]:
summary(model=model_cnn_2_brand,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNNReg_Net                               [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 32, 96, 160]    896                  True
├─BatchNorm2d: 1-2                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-3                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─Conv2d: 1-4                            [32, 32, 96, 160]    [32, 32, 96, 160]    9,248                True
├─BatchNorm2d: 1-5                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-6                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─MaxPool2d: 1-7                         [32, 32, 96, 160]    [32, 32, 48, 80]     --                   --
├─Conv2d: 1-8       

In [ ]:
main_kolesa(model_cnn_2_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_cnn_2_brand, 'Reg_CNN_brand')


Epoch: 1


100%|██████████| 97/97 [00:19<00:00,  5.10it/s]



Train set: Average loss: 2.0976, Accuracy: 15.37%
Val set: Average loss: 2.0732, Accuracy: 14.29%

Classification report:
               precision    recall  f1-score   support

         Audi       0.20      0.01      0.03        73
          BMW       0.11      0.17      0.14        90
    Chevrolet       0.16      0.67      0.26        79
      Hyundai       0.00      0.00      0.00        85
          Kia       0.33      0.12      0.17        77
        Lexus       0.11      0.12      0.11        95
Mercedes-Benz       0.00      0.00      0.00        81
       Toyota       0.09      0.07      0.08        85

     accuracy                           0.14       665
    macro avg       0.13      0.14      0.10       665
 weighted avg       0.12      0.14      0.10       665


Epoch: 2


100%|██████████| 97/97 [00:19<00:00,  4.90it/s]



Train set: Average loss: 2.0746, Accuracy: 17.60%
Val set: Average loss: 2.0500, Accuracy: 16.84%

Classification report:
               precision    recall  f1-score   support

         Audi       0.11      0.01      0.02        73
          BMW       0.16      0.22      0.19        90
    Chevrolet       0.15      0.70      0.25        79
      Hyundai       1.00      0.02      0.05        85
          Kia       0.21      0.21      0.21        77
        Lexus       0.19      0.15      0.17        95
Mercedes-Benz       0.33      0.01      0.02        81
       Toyota       0.16      0.04      0.06        85

     accuracy                           0.17       665
    macro avg       0.29      0.17      0.12       665
 weighted avg       0.29      0.17      0.12       665


Epoch: 3


...


Train set: Average loss: 1.2394, Accuracy: 46.79%
Val set: Average loss: 1.9829, Accuracy: 27.97%

Classification report:
               precision    recall  f1-score   support

         Audi       0.24      0.29      0.26        73
          BMW       0.30      0.54      0.39        90
    Chevrolet       0.58      0.27      0.37        79
      Hyundai       0.24      0.21      0.23        85
          Kia       0.28      0.26      0.27        77
        Lexus       0.29      0.27      0.28        95
Mercedes-Benz       0.22      0.30      0.25        81
       Toyota       0.21      0.08      0.12        85

     accuracy                           0.28       665
    macro avg       0.30      0.28      0.27       665
 weighted avg       0.29      0.28      0.27       665


Epoch: 29


100%|██████████| 97/97 [00:19<00:00,  5.10it/s]



Train set: Average loss: 1.3832, Accuracy: 47.02%
Val set: Average loss: 1.9554, Accuracy: 28.72%

Classification report:
               precision    recall  f1-score   support

         Audi       0.33      0.19      0.24        73
          BMW       0.39      0.43      0.41        90
    Chevrolet       0.48      0.35      0.41        79
      Hyundai       0.27      0.24      0.25        85
          Kia       0.28      0.32      0.30        77
        Lexus       0.22      0.25      0.23        95
Mercedes-Benz       0.26      0.35      0.29        81
       Toyota       0.16      0.15      0.16        85

     accuracy                           0.29       665
    macro avg       0.30      0.29      0.29       665
 weighted avg       0.30      0.29      0.29       665


Epoch: 30


100%|██████████| 97/97 [00:18<00:00,  5.16it/s]



Train set: Average loss: 1.2477, Accuracy: 48.47%
Val set: Average loss: 1.9180, Accuracy: 28.57%

Classification report:
               precision    recall  f1-score   support

         Audi       0.33      0.26      0.29        73
          BMW       0.31      0.53      0.40        90
    Chevrolet       0.67      0.23      0.34        79
      Hyundai       0.23      0.25      0.24        85
          Kia       0.29      0.31      0.30        77
        Lexus       0.23      0.29      0.26        95
Mercedes-Benz       0.27      0.32      0.29        81
       Toyota       0.18      0.07      0.10        85

     accuracy                           0.29       665
    macro avg       0.31      0.28      0.28       665
 weighted avg       0.31      0.29      0.28       665

Training is ended!
[OK] модель сохранена: models/Reg_CNN_brand.pt


И вернем дефолтное количество эпох

In [ ]:
CFG.num_epochs = 10

Вышло уже в разы лучше

## FineTuned AlexNet

Далее попробуем в качестве другого подхода взять уже ранее предобученную модель. Возьмем AlexNet

In [ ]:
from torchvision.models import alexnet
from torchvision.models import AlexNet_Weights

num_in_features = 9216  # Alex net после сверток выдает 256 каналов, а пуллинг сжимает картинку до 6 x 6
num_out_features = 2    # Sedan и SUV

CFG.lr = 1e-4 #learning rate понижаем для более корректного шага модели

#### Задача на тип кузова

In [ ]:
CFG.classes = all_body_types

model_alexnet_bt = alexnet(weights=AlexNet_Weights.DEFAULT) #DEFAULT - рекомендованный вариант
model_alexnet_bt.classifier = torch.nn.Linear(num_in_features, num_out_features) #alexnet.classifier по умолчанию выдает 1000 классов, а нам нужно 2, поэтому сокращаем

In [ ]:
summary(model=model_alexnet_bt,
        input_size=(32, 3, 96, 160), #картинкики, канал, высота и ширина
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
) #прогоняем фиктивный батч через модель

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
AlexNet                                  [32, 3, 96, 160]     [32, 2]              --                   True
├─Sequential: 1-1                        [32, 3, 96, 160]     [32, 256, 2, 4]      --                   True
│    └─Conv2d: 2-1                       [32, 3, 96, 160]     [32, 64, 23, 39]     23,296               True
│    └─ReLU: 2-2                         [32, 64, 23, 39]     [32, 64, 23, 39]     --                   --
│    └─MaxPool2d: 2-3                    [32, 64, 23, 39]     [32, 64, 11, 19]     --                   --
│    └─Conv2d: 2-4                       [32, 64, 11, 19]     [32, 192, 11, 19]    307,392              True
│    └─ReLU: 2-5                         [32, 192, 11, 19]    [32, 192, 11, 19]    --                   --
│    └─MaxPool2d: 2-6                    [32, 192, 11, 19]    [32, 192, 5, 9]      --                   --
│    └─Conv2d: 2-7    

In [ ]:
main_kolesa(model_alexnet_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_alexnet_bt, 'AlexNet_finetune_body_type')


Epoch: 1


  0%|          | 0/97 [00:00<?, ?it/s]


RuntimeError: Adaptive pool MPS: output sizes must be divisible by input sizes. Non-divisible input sizes are not implemented on MPS device yet. For now, you can manually transfer tensor to cpu in this case. Please refer to [this issue](https://github.com/pytorch/pytorch/issues/96056)

#### Задача на определение марки

In [ ]:
num_out_features = 8 # 8 разных марок
CFG.classes = all_brands

model_alexnet_brand = alexnet(weights=AlexNet_Weights.DEFAULT) #DEFAULT - рекомендованный вариант
model_alexnet_brand.classifier = torch.nn.Linear(num_in_features, num_out_features) #alexnet.classifier по умолчанию выдает 1000 классов, а нам нужно 8, поэтому сокращаем

In [ ]:
summary(model=model_alexnet_brand,
        input_size=(32, 3, 96, 160), #картинкики, канал, высота и ширина
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
) #прогоняем фиктивный батч через модель

In [ ]:
main_kolesa(model_alexnet_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_alexnet_brand, 'AlexNet_finetune_brand')

Доообучение прошло хорошо. Train accuracy выросло с 65 до 90%, на валидации с 70 до 78%.

Качество нессиметрично по классам - Седан распозднается стабильно лучше внедородника

Попробуем второй вариант (который был на семинаре) дообучать только верхние слои + классификатор.

Замороженные параметры (`requires_grad=False`) оптимизатор просто не обновляет.




#### Задача на тип кузова

In [ ]:
CFG.classes = all_body_types
num_out_features = 2

model_alexnet_frozen_bt = alexnet(weights=AlexNet_Weights.DEFAULT)

# замораживаем первые 6 параметров f
layers_to_freeze = 6
for i, (name, param) in enumerate(model_alexnet_frozen_bt.features.named_parameters()):
    if i < layers_to_freeze:
        param.requires_grad = False
    print(f'{name:30}{param.requires_grad}') #выводим имя параметра и его статус (будет использоваться или гет)
    #name:30 выравниваение параметров по ширине в 30 символов (чтобы аккуратно было)

model_alexnet_frozen_bt.classifier = torch.nn.Linear(num_in_features, num_out_features)

0.weight                      False
0.bias                        False
3.weight                      False
3.bias                        False
6.weight                      False
6.bias                        False
8.weight                      True
8.bias                        True
10.weight                     True
10.bias                       True


In [ ]:
main_kolesa(model_alexnet_frozen_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_alexnet_frozen_bt, 'AlexNet_finetune_frozen_body_type')


Epoch: 1


  0%|          | 0/97 [00:00<?, ?it/s]


RuntimeError: Adaptive pool MPS: output sizes must be divisible by input sizes. Non-divisible input sizes are not implemented on MPS device yet. For now, you can manually transfer tensor to cpu in this case. Please refer to [this issue](https://github.com/pytorch/pytorch/issues/96056)

#### Задача на определение марки

In [ ]:
CFG.classes = all_brands
num_out_features = 8

model_alexnet_frozen_brand = alexnet(weights=AlexNet_Weights.DEFAULT)

# замораживаем первые 6 параметров f
layers_to_freeze = 6
for i, (name, param) in enumerate(model_alexnet_frozen_brand.features.named_parameters()):
    if i < layers_to_freeze:
        param.requires_grad = False
    print(f'{name:30}{param.requires_grad}') #выводим имя параметра и его статус (будет использоваться или гет)
    #name:30 выравниваение параметров по ширине в 30 символов (чтобы аккуратно было)

model_alexnet_frozen_brand.classifier = torch.nn.Linear(num_in_features, num_out_features)

In [ ]:
main_kolesa(model_alexnet_frozen_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_alexnet_frozen_brand, 'AlexNet_finetune_frozen_brand')

Заморозка дала результат по всем метрикам: accuracy +2 п.п., macro-F1 +0.03, и что важно - подтянулся слабый класс «внедорожник» (F1 0.72 → 0.75).

## ResNet18

Возьмём более современную **ResNet18**.

Ее преимущество - остаточные связи, которые позволяют обучать глубокие сети без затухания градиента.

Дообучаем так же — через наш `main_kolesa`, заменив только финальный полносвязный слой.

In [ ]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

#### Задача на тип кузова

In [ ]:
CFG.classes = all_body_types
num_out_features = 2

model_resnet_bt = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_bt.fc = torch.nn.Linear(model_resnet_bt.fc.in_features, num_out_features)

In [ ]:
summary(model=model_resnet_bt,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [ ]:
main_kolesa(model_resnet_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_resnet_bt, 'ResNet18_finetune_body_type')


Epoch: 1


100%|██████████| 97/97 [00:22<00:00,  4.36it/s]



Train set: Average loss: 0.6797, Accuracy: 72.99%
Val set: Average loss: 0.3294, Accuracy: 83.61%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.89      0.87       418
         SUV       0.81      0.74      0.77       247

    accuracy                           0.84       665
   macro avg       0.83      0.82      0.82       665
weighted avg       0.83      0.84      0.83       665


Epoch: 2


100%|██████████| 97/97 [00:17<00:00,  5.67it/s]



Train set: Average loss: 0.4057, Accuracy: 85.98%
Val set: Average loss: 0.3828, Accuracy: 83.61%

Classification report:
              precision    recall  f1-score   support

       sedan       0.87      0.87      0.87       418
         SUV       0.78      0.78      0.78       247

    accuracy                           0.84       665
   macro avg       0.82      0.82      0.82       665
weighted avg       0.84      0.84      0.84       665


Epoch: 3


...


Train set: Average loss: 0.0487, Accuracy: 97.49%
Val set: Average loss: 0.3203, Accuracy: 89.17%

Classification report:
              precision    recall  f1-score   support

       sedan       0.91      0.92      0.91       418
         SUV       0.86      0.84      0.85       247

    accuracy                           0.89       665
   macro avg       0.89      0.88      0.88       665
weighted avg       0.89      0.89      0.89       665


Epoch: 9


100%|██████████| 97/97 [00:20<00:00,  4.83it/s]



Train set: Average loss: 0.0760, Accuracy: 97.94%
Val set: Average loss: 0.2873, Accuracy: 87.37%

Classification report:
              precision    recall  f1-score   support

       sedan       0.89      0.91      0.90       418
         SUV       0.84      0.81      0.83       247

    accuracy                           0.87       665
   macro avg       0.87      0.86      0.86       665
weighted avg       0.87      0.87      0.87       665


Epoch: 10


100%|██████████| 97/97 [00:21<00:00,  4.61it/s]



Train set: Average loss: 0.0479, Accuracy: 98.36%
Val set: Average loss: 0.4579, Accuracy: 87.82%

Classification report:
              precision    recall  f1-score   support

       sedan       0.90      0.91      0.90       418
         SUV       0.85      0.82      0.83       247

    accuracy                           0.88       665
   macro avg       0.87      0.87      0.87       665
weighted avg       0.88      0.88      0.88       665

Training is ended!
[OK] модель сохранена: models/ResNet18_finetune_body_type.pt


#### Задача на определение марки

In [ ]:
CFG.classes = all_brands
num_out_features = 8

model_resnet_brand = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_brand.fc = torch.nn.Linear(model_resnet_brand.fc.in_features, num_out_features)

In [ ]:
summary(model=model_resnet_brand,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [ ]:
main_kolesa(model_resnet_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_resnet_brand, 'ResNet18_finetune_brand')


Epoch: 1


100%|██████████| 97/97 [00:17<00:00,  5.66it/s]



Train set: Average loss: 1.6108, Accuracy: 26.07%
Val set: Average loss: 1.7420, Accuracy: 37.14%

Classification report:
               precision    recall  f1-score   support

         Audi       0.61      0.34      0.44        73
          BMW       0.71      0.36      0.47        90
    Chevrolet       0.39      0.75      0.51        79
      Hyundai       0.35      0.26      0.30        85
          Kia       0.29      0.32      0.31        77
        Lexus       0.32      0.26      0.29        95
Mercedes-Benz       0.29      0.58      0.39        81
       Toyota       0.32      0.14      0.20        85

     accuracy                           0.37       665
    macro avg       0.41      0.38      0.36       665
 weighted avg       0.41      0.37      0.36       665


Epoch: 2


100%|██████████| 97/97 [00:16<00:00,  5.76it/s]



Train set: Average loss: 1.1984, Accuracy: 50.02%
Val set: Average loss: 1.6863, Accuracy: 47.67%

Classification report:
               precision    recall  f1-score   support

         Audi       0.75      0.45      0.56        73
          BMW       0.61      0.78      0.69        90
    Chevrolet       0.60      0.48      0.54        79
      Hyundai       0.51      0.21      0.30        85
          Kia       0.39      0.35      0.37        77
        Lexus       0.44      0.55      0.49        95
Mercedes-Benz       0.42      0.58      0.48        81
       Toyota       0.29      0.38      0.33        85

     accuracy                           0.48       665
    macro avg       0.50      0.47      0.47       665
 weighted avg       0.50      0.48      0.47       665


Epoch: 3


...


Train set: Average loss: 0.2971, Accuracy: 88.66%
Val set: Average loss: 1.7923, Accuracy: 54.89%

Classification report:
               precision    recall  f1-score   support

         Audi       0.54      0.75      0.63        73
          BMW       0.64      0.72      0.68        90
    Chevrolet       0.67      0.63      0.65        79
      Hyundai       0.53      0.41      0.46        85
          Kia       0.48      0.60      0.53        77
        Lexus       0.56      0.44      0.49        95
Mercedes-Benz       0.46      0.59      0.52        81
       Toyota       0.51      0.28      0.36        85

     accuracy                           0.55       665
    macro avg       0.55      0.55      0.54       665
 weighted avg       0.55      0.55      0.54       665


Epoch: 9


100%|██████████| 97/97 [00:18<00:00,  5.18it/s]



Train set: Average loss: 0.6464, Accuracy: 89.08%
Val set: Average loss: 1.9045, Accuracy: 54.89%

Classification report:
               precision    recall  f1-score   support

         Audi       0.56      0.66      0.61        73
          BMW       0.70      0.71      0.70        90
    Chevrolet       0.78      0.57      0.66        79
      Hyundai       0.54      0.53      0.54        85
          Kia       0.57      0.39      0.46        77
        Lexus       0.45      0.66      0.54        95
Mercedes-Benz       0.60      0.47      0.53        81
       Toyota       0.35      0.38      0.36        85

     accuracy                           0.55       665
    macro avg       0.57      0.55      0.55       665
 weighted avg       0.57      0.55      0.55       665


Epoch: 10


100%|██████████| 97/97 [00:17<00:00,  5.62it/s]



Train set: Average loss: 0.0875, Accuracy: 92.49%
Val set: Average loss: 1.9049, Accuracy: 56.99%

Classification report:
               precision    recall  f1-score   support

         Audi       0.77      0.55      0.64        73
          BMW       0.76      0.70      0.73        90
    Chevrolet       0.50      0.82      0.62        79
      Hyundai       0.47      0.66      0.55        85
          Kia       0.59      0.47      0.52        77
        Lexus       0.63      0.54      0.58        95
Mercedes-Benz       0.56      0.53      0.54        81
       Toyota       0.39      0.29      0.34        85

     accuracy                           0.57       665
    macro avg       0.58      0.57      0.57       665
 weighted avg       0.58      0.57      0.57       665

Training is ended!
[OK] модель сохранена: models/ResNet18_finetune_brand.pt


  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ksa/CodeProjects/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

У нас получились пока все еще плохие результаты для задачи на марки. Модель переобучилась. Попробуем изменить гиперпараметры для ResNet.

Сделаем измененную main функцию

In [ ]:
def main_kolesa_new(model, train_df, val_df):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))
        # логируем архитектуру модели
        # https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})

    seed_everything(CFG.seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # Попробуем картинки побольше
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((320, 512)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((320, 512)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    # test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    # test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    # https://habr.com/ru/articles/988198/
    # добавим l2 регуляризацию (weight_decay)
    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr, weight_decay=3e-4)

    # добавим небольшой коэф. сглаживания, что может помочь с переобучением
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)

    print('Training is ended!')

Изменим learning rate и возьмем побольше эпох

In [ ]:
CFG.num_epochs = 15
CFG.lr = 3e-5

Также добавим dropout (регуляризацию) в модель.

### Задача на тип кузова

In [ ]:
CFG.classes = all_body_types
num_out_features = 2
CFG.test_batch_size = 64 # иначе не хватает памяти с дефолтным 512

# https://discuss.pytorch.org/t/resnet-last-layer-modification/33530
model_resnet_bt_reg = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_bt_reg.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model_resnet_bt_reg.fc.in_features, num_out_features)
) 

In [ ]:
summary(model=model_resnet_bt_reg,
        input_size=(32, 3, 320, 512),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 320, 512]    [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 320, 512]    [32, 64, 160, 256]   9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 160, 256]   [32, 64, 160, 256]   128                  True
├─ReLU: 1-3                              [32, 64, 160, 256]   [32, 64, 160, 256]   --                   --
├─MaxPool2d: 1-4                         [32, 64, 160, 256]   [32, 64, 80, 128]    --                   --
├─Sequential: 1-5                        [32, 64, 80, 128]    [32, 64, 80, 128]    --                   True
│    └─BasicBlock: 2-1                   [32, 64, 80, 128]    [32, 64, 80, 128]    --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 80, 128]    [32, 64, 80, 128]    36,864               True
│    │    └─BatchN

In [ ]:
main_kolesa_new(model_resnet_bt_reg, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_resnet_bt_reg, 'ResNet18_reg_finetune_bt')


Epoch: 1


  0%|          | 0/97 [00:11<?, ?it/s]


KeyboardInterrupt: 

### Задача на марки

In [ ]:
CFG.classes = all_brands
num_out_features = 8
# https://discuss.pytorch.org/t/resnet-last-layer-modification/33530
model_resnet_brand_reg = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_brand_reg.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model_resnet_brand.fc.in_features, num_out_features)
) 

In [ ]:
summary(model=model_resnet_brand_reg,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [ ]:
main_kolesa_new(model_resnet_brand_reg, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_resnet_brand_reg, 'ResNet18_reg_finetune_brand')


Epoch: 1


100%|██████████| 97/97 [01:25<00:00,  1.14it/s]



Train set: Average loss: 1.2974, Accuracy: 44.02%
Val set: Average loss: 1.2707, Accuracy: 58.95%

Classification report:
               precision    recall  f1-score   support

         Audi       0.75      0.66      0.70        73
          BMW       0.76      0.88      0.81        90
    Chevrolet       0.48      0.87      0.62        79
      Hyundai       0.52      0.62      0.57        85
          Kia       0.59      0.39      0.47        77
        Lexus       0.52      0.48      0.50        95
Mercedes-Benz       0.61      0.68      0.64        81
       Toyota       0.52      0.14      0.22        85

     accuracy                           0.59       665
    macro avg       0.59      0.59      0.57       665
 weighted avg       0.59      0.59      0.57       665


Epoch: 2


100%|██████████| 97/97 [03:52<00:00,  2.39s/it]



Train set: Average loss: 0.9791, Accuracy: 66.65%
Val set: Average loss: 1.0123, Accuracy: 23.61%

Classification report:
               precision    recall  f1-score   support

         Audi       0.67      0.19      0.30        73
          BMW       0.80      0.18      0.29        90
    Chevrolet       0.57      0.15      0.24        79
      Hyundai       0.27      0.22      0.24        85
          Kia       0.13      0.78      0.22        77
        Lexus       0.53      0.19      0.28        95
Mercedes-Benz       1.00      0.16      0.28        81
       Toyota       0.71      0.06      0.11        85

     accuracy                           0.24       665
    macro avg       0.58      0.24      0.24       665
 weighted avg       0.59      0.24      0.24       665


Epoch: 3


...


Train set: Average loss: 0.3238, Accuracy: 99.19%
Val set: Average loss: 0.7005, Accuracy: 83.76%

Classification report:
               precision    recall  f1-score   support

         Audi       0.88      0.82      0.85        73
          BMW       0.90      0.90      0.90        90
    Chevrolet       0.84      0.91      0.87        79
      Hyundai       0.68      0.92      0.78        85
          Kia       0.90      0.81      0.85        77
        Lexus       0.92      0.74      0.82        95
Mercedes-Benz       0.85      0.88      0.86        81
       Toyota       0.81      0.74      0.77        85

     accuracy                           0.84       665
    macro avg       0.85      0.84      0.84       665
 weighted avg       0.85      0.84      0.84       665


Epoch: 14


100%|██████████| 97/97 [01:08<00:00,  1.42it/s]



Train set: Average loss: 0.3403, Accuracy: 99.26%
Val set: Average loss: 0.7004, Accuracy: 84.51%

Classification report:
               precision    recall  f1-score   support

         Audi       0.87      0.85      0.86        73
          BMW       0.88      0.90      0.89        90
    Chevrolet       0.77      0.92      0.84        79
      Hyundai       0.73      0.88      0.80        85
          Kia       0.90      0.82      0.86        77
        Lexus       0.89      0.80      0.84        95
Mercedes-Benz       0.88      0.88      0.88        81
       Toyota       0.90      0.72      0.80        85

     accuracy                           0.85       665
    macro avg       0.85      0.85      0.85       665
 weighted avg       0.85      0.85      0.85       665


Epoch: 15


100%|██████████| 97/97 [01:15<00:00,  1.28it/s]



Train set: Average loss: 0.3202, Accuracy: 99.65%
Val set: Average loss: 0.6829, Accuracy: 85.71%

Classification report:
               precision    recall  f1-score   support

         Audi       0.87      0.82      0.85        73
          BMW       0.89      0.90      0.90        90
    Chevrolet       0.79      0.95      0.86        79
      Hyundai       0.79      0.91      0.85        85
          Kia       0.89      0.81      0.84        77
        Lexus       0.89      0.83      0.86        95
Mercedes-Benz       0.88      0.89      0.88        81
       Toyota       0.89      0.75      0.82        85

     accuracy                           0.86       665
    macro avg       0.86      0.86      0.86       665
 weighted avg       0.86      0.86      0.86       665

Training is ended!
[OK] модель сохранена: models/ResNet18_reg_finetune_brand.pt


Модель переобучилась но качество сильно лучше. Добавим scheduler для снижения шага модели с каждой эпохой + добавим больше эпох и смягчим dropout

In [ ]:
def main_kolesa_new(model, train_df, val_df):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))
        # логируем архитектуру модели
        # https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})

    seed_everything(CFG.seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # Попробуем картинки побольше
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((320, 512)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((320, 512)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    # test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    # test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    # https://habr.com/ru/articles/988198/
    # добавим l2 регуляризацию (weight_decay)
    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr, weight_decay=3e-4)
    
    # https://discuss.pytorch.org/t/how-to-choose-learning-rate-and-scheduler/174319/4
    # https://www.kaggle.com/code/greatcodes/pytorch-cnn-resnet18-cifar10?scriptVersionId=42951150&cellId=12
    # добавим scheduler чтобы наш lr не был статическим
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    # добавим небольшой коэф. сглаживания, что может помочь с переобучением
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)
        scheduler.step()

    print('Training is ended!')

In [ ]:
CFG.num_epochs = 20
CFG.lr = 3e-5
CFG.classes = all_brands
CFG.test_batch_size = 64 # иначе не хватает памяти с дефолтным 512
num_out_features = 8
# https://discuss.pytorch.org/t/resnet-last-layer-modification/33530
model_resnet_brand_reg_v2 = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_brand_reg_v2.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model_resnet_brand_reg_v2.fc.in_features, num_out_features)
) 

In [ ]:
summary(model=model_resnet_brand_reg_v2,
        input_size=(32, 3, 320, 512),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 320, 512]    [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 320, 512]    [32, 64, 160, 256]   9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 160, 256]   [32, 64, 160, 256]   128                  True
├─ReLU: 1-3                              [32, 64, 160, 256]   [32, 64, 160, 256]   --                   --
├─MaxPool2d: 1-4                         [32, 64, 160, 256]   [32, 64, 80, 128]    --                   --
├─Sequential: 1-5                        [32, 64, 80, 128]    [32, 64, 80, 128]    --                   True
│    └─BasicBlock: 2-1                   [32, 64, 80, 128]    [32, 64, 80, 128]    --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 80, 128]    [32, 64, 80, 128]    36,864               True
│    │    └─BatchN

In [ ]:
main_kolesa_new(model_resnet_brand_reg_v2, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_resnet_brand_reg_v2, 'ResNet18_reg_finetune_brand_v2')


Epoch: 1


100%|██████████| 97/97 [01:02<00:00,  1.56it/s]



Train set: Average loss: 1.8958, Accuracy: 23.11%
Val set: Average loss: 1.8351, Accuracy: 43.61%

Classification report:
               precision    recall  f1-score   support

         Audi       0.69      0.55      0.61        73
          BMW       0.46      0.84      0.60        90
    Chevrolet       0.45      0.57      0.50        79
      Hyundai       0.37      0.31      0.33        85
          Kia       0.41      0.43      0.42        77
        Lexus       0.28      0.36      0.31        95
Mercedes-Benz       0.60      0.33      0.43        81
       Toyota       0.43      0.11      0.17        85

     accuracy                           0.44       665
    macro avg       0.46      0.44      0.42       665
 weighted avg       0.45      0.44      0.42       665


Epoch: 2


100%|██████████| 97/97 [01:03<00:00,  1.53it/s]



Train set: Average loss: 1.3273, Accuracy: 50.27%
Val set: Average loss: 1.3157, Accuracy: 65.56%

Classification report:
               precision    recall  f1-score   support

         Audi       0.70      0.75      0.72        73
          BMW       0.86      0.83      0.85        90
    Chevrolet       0.73      0.81      0.77        79
      Hyundai       0.55      0.65      0.59        85
          Kia       0.71      0.53      0.61        77
        Lexus       0.44      0.72      0.54        95
Mercedes-Benz       0.82      0.68      0.74        81
       Toyota       0.77      0.27      0.40        85

     accuracy                           0.66       665
    macro avg       0.70      0.66      0.65       665
 weighted avg       0.69      0.66      0.65       665


Epoch: 3


...


Train set: Average loss: 0.3548, Accuracy: 97.81%
Val set: Average loss: 0.7807, Accuracy: 85.71%

Classification report:
               precision    recall  f1-score   support

         Audi       0.86      0.82      0.84        73
          BMW       0.93      0.90      0.92        90
    Chevrolet       0.86      0.90      0.88        79
      Hyundai       0.77      0.91      0.83        85
          Kia       0.90      0.84      0.87        77
        Lexus       0.80      0.86      0.83        95
Mercedes-Benz       0.89      0.86      0.88        81
       Toyota       0.89      0.75      0.82        85

     accuracy                           0.86       665
    macro avg       0.86      0.86      0.86       665
 weighted avg       0.86      0.86      0.86       665


Epoch: 19


100%|██████████| 97/97 [01:00<00:00,  1.60it/s]



Train set: Average loss: 0.3866, Accuracy: 97.71%
Val set: Average loss: 0.7781, Accuracy: 85.71%

Classification report:
               precision    recall  f1-score   support

         Audi       0.85      0.82      0.83        73
          BMW       0.93      0.90      0.92        90
    Chevrolet       0.86      0.90      0.88        79
      Hyundai       0.76      0.94      0.84        85
          Kia       0.93      0.83      0.88        77
        Lexus       0.82      0.86      0.84        95
Mercedes-Benz       0.88      0.84      0.86        81
       Toyota       0.88      0.75      0.81        85

     accuracy                           0.86       665
    macro avg       0.86      0.86      0.86       665
 weighted avg       0.86      0.86      0.86       665


Epoch: 20


100%|██████████| 97/97 [01:00<00:00,  1.60it/s]



Train set: Average loss: 0.3690, Accuracy: 97.68%
Val set: Average loss: 0.7621, Accuracy: 85.71%

Classification report:
               precision    recall  f1-score   support

         Audi       0.85      0.82      0.83        73
          BMW       0.93      0.90      0.92        90
    Chevrolet       0.86      0.90      0.88        79
      Hyundai       0.75      0.94      0.84        85
          Kia       0.93      0.82      0.87        77
        Lexus       0.82      0.86      0.84        95
Mercedes-Benz       0.87      0.85      0.86        81
       Toyota       0.90      0.75      0.82        85

     accuracy                           0.86       665
    macro avg       0.86      0.86      0.86       665
 weighted avg       0.86      0.86      0.86       665

Training is ended!
[OK] модель сохранена: models/ResNet18_reg_finetune_brand_v2.pt


ResNet18 показала лучший результат среди всех архитектур: на валидации accuracy 87.5% и macro-F1 0.86, с уверенным распознаванием обоих классов (sedan F1 0.90, SUV F1 0.82).

Также можно отметить, что модель обучилась быстро, так как к 3-ей эпохе вышло на плато, превосходя значительно остальные модели.

На последней эпохе есть переобучение (98% accuracy на тесте).

## Сравнение и выводы